In [ ]:
import json
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import lightgbm as lgb

# ==========================================
# 🚀 1. 경로 설정 및 디렉토리 생성
# ==========================================
DATA_DIR = "./data/data/processed"
os.makedirs("./model", exist_ok=True)
os.makedirs("./test", exist_ok=True)

print("=" * 50)
print("🚀 1. 데이터 로드 및 전처리 시작")
print("=" * 50)

# ==========================================
# 🚀 2. 데이터 로드 (정형 데이터, 시퀀스 텐서, 타겟 라벨)
# ==========================================
# 2.1 정형(Tabular) 데이터 로드 (CSV 파일)
csv_path = os.path.join(DATA_DIR, "customer_features_20181101_v1.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(DATA_DIR, "train_20181101_v1.csv")

df_ml = pd.read_csv(csv_path)
print(f"정형 데이터 shape: {df_ml.shape}")

# 2.2 시퀀스 텐서(NPY) 및 타겟 라벨 로드 (3번 파일, 2번 파일 반영)
seq_path = os.path.join(DATA_DIR, "converted_sequence_tensor.npy")
if not os.path.exists(seq_path):
    seq_path = os.path.join(DATA_DIR, "dl_sequence_tensor.npy")

if os.path.exists(seq_path):
    X_seq = np.load(seq_path)
    print(f"시퀀스 텐서(NPY) 로드 완료: shape = {X_seq.shape}")
else:
    raise FileNotFoundError(f"지정한 경로에 시퀀스 파일이 없습니다: {seq_path}")

# 타겟 변수 설정 (df_ml에서 이탈 여부 컬럼 동적 추출)
target_col = "churn" if "churn" in df_ml.columns else ("is_churn" if "is_churn" in df_ml.columns else df_ml.columns[-1])
y = df_ml[target_col].values
print(f"타겟 라벨(DataFrame) 로드 완료: 총 {len(y):,}개 샘플 (컬럼명: {target_col})")

# ==========================================
# 🚀 3. LightGBM용 정형 데이터 피처 분리 및 타입 정제
# ==========================================
drop_cols = [
    target_col,
    "userId",
    "page_sequence",
    "session_item_sequence",
    "padded_sequence",
]
feature_cols = [col for col in df_ml.columns if col not in drop_cols]
X_tab = df_ml[feature_cols].copy()

# 범주형 변수 타입 변환 (LightGBM 최적화)
cat_cols = ["gender", "level", "last_page"]
for col in cat_cols:
    if col in X_tab.columns:
        X_tab[col] = X_tab[col].astype("category")

# 수치형 변수 강제 변환
for col in X_tab.columns:
    if X_tab[col].dtype == "object" and col not in cat_cols:
        X_tab[col] = pd.to_numeric(X_tab[col], errors="coerce").fillna(0.0)

print(f"ML 정형 데이터 shape: {X_tab.shape}")
print(f"DL 시퀀스 텐서 shape: {X_seq.shape}")
print(f"타겟 라벨 shape: {y.shape}")

# ==========================================
# 🚀 4. 데이터셋 분할 (Train / Test Split)
# ==========================================
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

X_tab_train, X_tab_test = X_tab.iloc[train_idx], X_tab.iloc[test_idx]
X_seq_train, X_seq_test = X_seq[train_idx], X_seq[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"학습 데이터 수: {len(y_train):,}명, 테스트 데이터 수: {len(y_test):,}명")

# ==========================================
# 🚀 5. 정형 모델 학습: LightGBM (Tabular Branch)
# ==========================================
print("\n" + "=" * 50)
print("🚀 2. 정형 모델 학습: LightGBM (Tabular Branch)")
print("=" * 50)

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "max_depth": 3,
    "num_leaves": 7,
    "min_child_samples": 50,
    "reg_alpha": 1.0,
    "reg_lambda": 1.0,
    "n_estimators": 500,
    "random_state": 42,
    "verbose": -1,
}

lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(
    X_tab_train,
    y_train,
    eval_set=[(X_tab_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)],
)

lgb_preds_proba = lgb_model.predict_proba(X_tab_test)[:, 1]
lgb_auc = roc_auc_score(y_test, lgb_preds_proba)
print(f"🚀 [LightGBM 단독] Test ROC-AUC Score: {lgb_auc:.4f}")

# ==========================================
# 🚀 6. 시퀀스 모델 학습: PyTorch LSTM (Sequence Branch)
# ==========================================
print("\n" + "=" * 50)
print("🚀 3. 시퀀스 모델 학습: PyTorch LSTM (Sequence Branch)")
print("=" * 50)

BATCH_SIZE = 64
X_tr, X_val, y_tr, y_val = train_test_split(
    X_seq_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

train_dataset = TensorDataset(
    torch.tensor(X_tr, dtype=torch.long),
    torch.tensor(y_tr, dtype=torch.float32),
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.long),
    torch.tensor(y_val, dtype=torch.float32),
)
test_dataset = TensorDataset(
    torch.tensor(X_seq_test, dtype=torch.long),
    torch.tensor(y_test, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# PyTorch LSTM 모델 정의 (정통 LSTM 구조)
class ChurnLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hn, cn) = self.lstm(embedded)
        out = hn[-1]
        out = self.fc(out)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lstm_model = ChurnLSTM(vocab_size=50, embedding_dim=32, hidden_dim=64, num_layers=1).to(device)

# 클래스 불균형 해소를 위한 WeightedRandomSampler 적용
class_counts = np.bincount(y_tr.astype(int))
class_weights = 1.0 / class_counts
sample_weights = class_weights[y_tr.astype(int)]

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)

# Focal Loss 정의
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        return torch.mean(focal_loss)

EPOCHS = 20
patience = 4
best_val_loss = float("inf")
patience_counter = 0

criterion = FocalLoss(alpha=0.75, gamma=2.0)
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

for epoch in range(EPOCHS):
    # 1. Train 단계
    lstm_model.train()
    train_loss = 0
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = lstm_model(seqs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # 2. Validation 단계
    lstm_model.eval()
    val_loss = 0
    with torch.no_grad():
        for seqs, labels in val_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            outputs = lstm_model(seqs)
            loss = criterion(outputs.squeeze(), labels)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    scheduler.step(avg_val_loss)
    
    # 3. 조기 종료(Early Stopping) 체크
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(lstm_model.state_dict(), "best_lstm.pt")
    else:
        patience_counter += 1
        print(f"  ⚠️ Validation Loss 개선 없음 ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"🛑 조기 종료 (Early Stopping) 발동! (Epoch {epoch+1})")
            lstm_model.load_state_dict(torch.load("best_lstm.pt"))
            break

# 테스트셋 예측 수행
lstm_model.eval()
lstm_preds_list = []
with torch.no_grad():
    for seqs, _ in test_loader:
        seqs = seqs.to(device)
        outputs = torch.sigmoid(lstm_model(seqs))
        lstm_preds_list.extend(outputs.cpu().numpy())

lstm_preds_proba = np.array(lstm_preds_list).ravel()
lstm_auc = roc_auc_score(y_test, lstm_preds_proba)
print(f"🚀 [LSTM 단독] Test ROC-AUC Score: {lstm_auc:.4f}")


🚀 1. 데이터 로드 및 전처리 시작
정형 데이터 shape: (18139, 16)
시퀀스 텐서(NPY) 로드 완료: shape = (18139, 50)
타겟 라벨(DataFrame) 로드 완료: 총 18,139개 샘플 (컬럼명: churn)
ML 정형 데이터 shape: (18139, 15)
DL 시퀀스 텐서 shape: (18139, 50)
타겟 라벨 shape: (18139,)
학습 데이터 수: 14,511명, 테스트 데이터 수: 3,628명

🚀 2. 정형 모델 학습: LightGBM (Tabular Branch)


c:\Users\playdata2\miniconda3\envs\agent\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.715069
[200]	valid_0's auc: 0.715651
Early stopping, best iteration is:
[153]	valid_0's auc: 0.716271
🚀 [LightGBM 단독] Test ROC-AUC Score: 0.7163

🚀 3. 시퀀스 모델 학습: PyTorch LSTM (Sequence Branch)
Epoch [1/20] | Train Loss: 0.0936 | Val Loss: 0.0950
Epoch [2/20] | Train Loss: 0.0845 | Val Loss: 0.0828
Epoch [3/20] | Train Loss: 0.0813 | Val Loss: 0.0862
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [4/20] | Train Loss: 0.0778 | Val Loss: 0.0822
Epoch [5/20] | Train Loss: 0.0749 | Val Loss: 0.0783
Epoch [6/20] | Train Loss: 0.0750 | Val Loss: 0.0762
Epoch [7/20] | Train Loss: 0.0747 | Val Loss: 0.0814
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [8/20] | Train Loss: 0.0746 | Val Loss: 0.0775
  ⚠️ Validation Loss 개선 없음 (2/4)
Epoch [9/20] | Train Loss: 0.0740 | Val Loss: 0.0801
  ⚠️ Validation Loss 개선 없음 (3/4)
Epoch [10/20] | Train Loss: 0.0740 | Val Loss: 0.0792
  ⚠️ Validation Loss 개선 없음 (4/4)
🛑 조기 종료 (Early Stopping) 발동

FileNotFoundError: [Errno 2] No such file or directory: './모델/lgb_model.pkl'

In [4]:

# ==========================================
# 🚀 7. 하이브리드 앙상블 (Late Fusion) 및 최종 평가
# ==========================================
print("\n" + "=" * 50)
print("🚀 4. 하이브리드 앙상블 (Late Fusion) 및 최종 평가")
print("=" * 50)

y_true = np.array(y_test).ravel()
best_auc = 0
best_w_lgb = 0.8
best_w_lstm = 0.2

for w in np.linspace(0.70, 0.90, 21):
    w_l = w
    w_s = 1.0 - w
    temp_preds = (w_l * lgb_preds_proba) + (w_s * lstm_preds_proba)
    temp_auc = roc_auc_score(y_true, temp_preds)
    if temp_auc > best_auc:
        best_auc = temp_auc
        best_w_lgb = w_l
        best_w_lstm = w_s

hybrid_preds_proba = (best_w_lgb * lgb_preds_proba) + (
    best_w_lstm * lstm_preds_proba
)

print(f" - LightGBM Weight: {best_w_lgb:.2f}")
print(f" - LSTM Weight: {best_w_lstm:.2f}")
print(f"🏆 [Hybrid Model] 최적 앙상블 Test ROC-AUC Score: {best_auc:.4f}")
print("-" * 50)

# --- 👇 이 부분을 추가/변경하여 최적 임계값 적용 ---
from sklearn.metrics import precision_recall_curve

# 1. Precision-Recall 곡선을 이용한 최적 임계값 탐색
precisions, recalls, thresholds = precision_recall_curve(y_true, hybrid_preds_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"🏆 F1-Score 기준 최적 임계값: {best_threshold:.4f}")

# 2. 최적 임계값으로 이탈 여부 판정 및 분류 리포트 출력
optimal_preds_threshold = (hybrid_preds_proba >= best_threshold).astype(int)
print(classification_report(y_true, optimal_preds_threshold))
# ----------------------------------------------------

# ==========================================
# 🚀 8. 결과 및 모델 아티팩트 저장
# ==========================================
joblib.dump(lgb_model, "./model/lgb_model.pkl")
torch.save(lstm_model.state_dict(), "./model/lstm_model_weights.pth")

np.save("./test/test_y_true.npy", y_true)
np.save("./test/X_test_sequence.npy", X_seq_test)
np.save("./test/hybrid_preds_proba.npy", hybrid_preds_proba)

hybrid_config = {
    "lgb_weight": float(best_w_lgb),
    "lstm_weight": float(best_w_lstm),
}
with open("./model/hybrid_config.json", "w", encoding="utf-8") as f:
    json.dump(hybrid_config, f, ensure_ascii=False, indent=4)

print("모든 모델과 평가 파일이 성공적으로 저장되었습니다!")


🚀 4. 하이브리드 앙상블 (Late Fusion) 및 최종 평가
 - LightGBM Weight: 0.70
 - LSTM Weight: 0.30
🏆 [Hybrid Model] 최적 앙상블 Test ROC-AUC Score: 0.8899
--------------------------------------------------
🏆 F1-Score 기준 최적 임계값: 0.2196
              precision    recall  f1-score   support

           0       0.98      0.89      0.93      3421
           1       0.25      0.63      0.36       207

    accuracy                           0.87      3628
   macro avg       0.61      0.76      0.64      3628
weighted avg       0.93      0.87      0.90      3628

모든 모델과 평가 파일이 성공적으로 저장되었습니다!
